In [ ]:
from pyfastpfor import *
import numpy as np
# Get the list of all codecs
# getCodecList()

In [ ]:
# scratch work, just one channel...
#   somehow this can be extended to do each channel... 
#   associate it with that timestamp
#   or, we can take the CCD route and align all measurements against a common time axis
#       sounds slower loading possibility, but probably best compression? 
import asammdf
sample_data_path = '../sample_data/sample_data.mf4'
channel_name = '1000ms_9'  # single example
with asammdf.MDF(sample_data_path) as mfil:
    sgl = mfil.select([channel_name])[0]
    vals = sgl.samples
    tmss = sgl.timestamps
# can we convert timestamp into us?
#   ie, will this be safe? not 100% sure 
#   something to ask MDF expert... lib author
# lets do it at first :) s * 1000
# tmss *= (1000 * 1000)  # s to us
tmss *= 1000  # s to ms
tmss = tmss.astype(np.uint64)  # TODO 64bit cannot be done with fastpfor as is, 
# need "turbopfor" --> see notes.txt

# compression is only done on uint, 
#   so TODO we need to handle negative values
#   allegedly this can be done with "zigzag encoding"?
#       which keeps the magnitudes of the values small
#       without it, the value of negative int will be large uint
#       and will not compress as well as possible

# trying zigzag --> see below
# vals = vals.astype(np.uint32)
vals.dtype, tmss.dtype

In [ ]:

tmss_orig = tmss.copy()
vals_orig = vals.copy()

In [ ]:
# zigzag coding --> big difference for negative ints!
#   TODO it is done under the hood with turbopfor library?
vals = (vals<<1)^(vals>>31)
vals = vals.astype(np.uint32)
# vals.dtype

In [ ]:
# # is this the same thing? yes, ok good
# (vals == (vals.astype(np.uint32).astype(np.int32))).all()

In [ ]:
# codec --> TODO which one is "best" for each dtype, inteval, ... etc?
codec = getCodec('simdbinarypacking')
# to compress, 
# positional arguments:
#   input array, 
#   size of array
#   buffer for compressed data (of the same type? does it matter?)
#   size (element count) of buffer --> so i guess it doesnt matter?
buffer_vals = np.zeros(shape=vals.shape[0]+100, dtype=np.uint32)  # TODO: uint variant of dtype? dtype=vals.dtype)
# compress
compSize_vals = codec.encodeArray(vals, int(len(vals)), buffer_vals, int(len(buffer_vals)))
buffer_vals[:compSize_vals].shape, vals.shape

In [ ]:
# take differential of time --> OK since we are always ascending in time, as per MDF standard
tmss = np.diff(tmss, prepend=0).astype(np.uint32)
# double differential of time would be more, 
#   any reason not to?
tmss = np.diff(tmss, prepend=0).astype(np.uint32)
tmss

In [ ]:
codec = getCodec('simdbinarypacking')
buffer_tmss = np.zeros(shape=tmss.shape[0]+100, dtype=np.uint32)
compSize_tmss = codec.encodeArray(tmss, int(len(tmss)), buffer_tmss, int(len(buffer_tmss)))
buffer_tmss[:compSize_tmss].shape, tmss.shape

In [ ]:
# "combined ratio" could be both... somehow...
f'simple example compression ratio: {((compSize_tmss + compSize_vals) / (len(tmss) + len(vals))):.3f}'

In [ ]:
# what about just zlibbing each transformed one...
#   large consistent integers seem to go better with zlib, 
#   maybe because of run length coding?
#   may be slower to decompress?
import zlib
(
zlib.compress(vals, level=9).__sizeof__() / vals.__sizeof__(),  # worse than fastpfor 
zlib.compress(tmss, level=9).__sizeof__() / tmss.__sizeof__(),  # better than fastpfor
)

In [ ]:
# what about zlib with no transformation...
import zlib
(
zlib.compress(vals_orig, level=9).__sizeof__() / vals.__sizeof__(),  # worse than fastpfor 
zlib.compress(tmss_orig, level=9).__sizeof__() / tmss.__sizeof__(),  # better than fastpfor
)

In [ ]:
# scratch below

In [ ]:
getCodecList()

In [ ]:
# original examples

In [ ]:
arrSize = 128 * 32
maxVal = 2048
# 1. Example without data differencing

# All arrays the library use must be contiguous-memory C-style numpy arrays
inp = np.array(np.random.randint(0, maxVal, arrSize), dtype = np.uint32, order = 'C')
inpCompDecomp = np.zeros(arrSize, dtype = np.uint32, order = 'C')

# To be on the safe side, let's reserve plenty of additional memory:
# sometimes the size of compressed data is not smaller than the size 
# of the original one
inpComp = np.zeros(arrSize + 1024, dtype = np.uint32, order = 'C')

# Obtain a codec by name
codec = getCodec('simdbinarypacking')

# Compress data
compSize = codec.encodeArray(inp, arrSize, inpComp, len(inpComp))
 
print('Compression ratio: %g' % (float(compSize)/arrSize))

# Decompress data
assert(arrSize == codec.decodeArray(inpComp, compSize, inpCompDecomp, arrSize))
assert(np.all(inpCompDecomp == inp))

In [ ]:
inp

In [ ]:
compSize

In [ ]:
arrSize

In [ ]:
inpComp[:compSize+3]

In [ ]:
arrSize = 128 * 32
maxVal = 1024 * 1024 * 1024 * 2

# 2. Example with slower data differencing

# All arrays the library use must be contiguous-memory C-style numpy arrays
inp = np.array(np.random.randint(0, maxVal, arrSize), dtype = np.uint32, order = 'C')
inpCompDecomp = np.zeros(arrSize, dtype = np.uint32, order = 'C')

inp.sort()
inpCopy = np.array(inp, copy = True, dtype = np.uint32, order = 'C')

# To be on the safe side, let's reserve plenty of additional memory:
# sometimes the size of compressed data is not smaller than the size 
# of the original one
inpComp = np.zeros(arrSize + 1024, dtype = np.uint32, order = 'C')

# Carry out dafa differencing to convert a sorted sequence of large numbers
# into a sequence of small numbers (differences between adjacent numbers)
delta1(inpCopy, arrSize)


# Obtain a codec by name
codec = getCodec('simdbinarypacking')

# Compress data
compSize = codec.encodeArray(inpCopy, arrSize, inpComp, len(inpComp))
 
print('Compression ratio: %g' % (float(compSize)/arrSize))

# Decompress data
assert(arrSize == codec.decodeArray(inpComp, compSize, inpCompDecomp, arrSize))
# Reverse differencing by computing the prefix sum
prefixSum1(inpCompDecomp, arrSize)

assert(np.all(inpCompDecomp == inp))

In [ ]:
arrSize = 128 * 32
maxVal = 1024 * 1024 * 1024 * 2

# 3. Example with faster but coarser data differencing

# All arrays the library use must be contiguous-memory C-style numpy arrays
inp = np.array(np.random.randint(0, maxVal, arrSize), dtype = np.uint32, order = 'C')
inpCompDecomp = np.zeros(arrSize, dtype = np.uint32, order = 'C')

inp.sort()
inpCopy = np.array(inp, copy = True, dtype = np.uint32, order = 'C')

# To be on the safe side, let's reserve plenty of additional memory:
# sometimes the size of compressed data is not smaller than the size 
# of the original one
inpComp = np.zeros(arrSize + 1024, dtype = np.uint32, order = 'C')

# Carry out dafa differencing to convert a sorted sequence of large numbers
# into a sequence of small numbers (differences between numbers that are 4 indices apart)
delta4(inpCopy, arrSize)


# Obtain a codec by name
codec = getCodec('simdbinarypacking')

# Compress data
compSize = codec.encodeArray(inpCopy, arrSize, inpComp, len(inpComp))
 
print('Compression ratio: %g' % (float(compSize)/arrSize))

# Decompress data
assert(arrSize == codec.decodeArray(inpComp, compSize, inpCompDecomp, arrSize))
# Reverse differencing by computing the prefix sum
prefixSum4(inpCompDecomp, arrSize)

assert(np.all(inpCompDecomp == inp))